# 🧪 W7-D3 概念实验：任务编排引擎到底在解决什么问题？

> 配套阅读：`第7周-Day3-任务编排与工作流.md`（工具编排 Level 1-4、Cron、消息路由在那边）
> 本 notebook 用 4 个实验回答：**为什么需要 DAG、为什么拓扑序是对的、失败重试值不值、并行能省多少时间。**
>
> 实验环境：纯标准库 + numpy/matplotlib 模拟，单 cell 秒级完成。


## 实验 1：把「商铺营业」写成 DAG —— 依赖声明 + Kahn 拓扑排序

人工排顺序容易漏依赖；把依赖显式声明成有向无环图（DAG），由拓扑排序保证「每个任务的所有前置都已完成」。
实现一个最小工作流引擎，并验证它能检测出循环依赖。


In [ ]:
from dataclasses import dataclass, field

@dataclass
class Task:
    name: str
    deps: list = field(default_factory=list)
    duration_s: int = 1

class Workflow:
    def __init__(self, tasks):
        self.tasks = {t.name: t for t in tasks}

    def topo_order(self):
        """Kahn 算法：入度为 0 的任务先执行，完成后削减后继入度"""
        indeg = {n: len(t.deps) for n, t in self.tasks.items()}
        succ = {n: [] for n in self.tasks}
        for n, t in self.tasks.items():
            for d in t.deps:
                succ[d].append(n)
        queue = sorted(n for n, d in indeg.items() if d == 0)
        order = []
        while queue:
            n = queue.pop(0)
            order.append(n)
            for s in sorted(succ[n]):
                indeg[s] -= 1
                if indeg[s] == 0:
                    queue.append(s)
        if len(order) != len(self.tasks):
            raise ValueError(f"检测到循环依赖！已排序: {order}")
        return order

shop = Workflow([
    Task("店员打卡", [], 1),
    Task("清洁", ["店员打卡"], 3),
    Task("商品检查", ["店员打卡"], 2),
    Task("开业", ["清洁", "商品检查"], 1),
    Task("订单处理", ["开业"], 8),
    Task("结算", ["开业"], 2),
    Task("数据统计", ["订单处理", "结算"], 2),
])
print("商铺营业拓扑序：")
for i, name in enumerate(shop.topo_order(), 1):
    t = shop.tasks[name]
    print(f"  {i}. {name:<6} (前置: {t.deps or '无'}, 耗时 {t.duration_s}s)")

# 验证拓扑序合法性：任意任务执行时其依赖都已出现过
order = shop.topo_order()
seen = set(); ok = True
for n in order:
    if not set(shop.tasks[n].deps) <= seen: ok = False
    seen.add(n)
print(f"\n拓扑序合法性校验：{'PASS' if ok else 'FAIL'}")

# 环检测演示
try:
    Workflow([Task("A", ["B"]), Task("B", ["A"])]).topo_order()
except ValueError as e:
    print(f"循环依赖被正确拒绝：{e}")

## 实验 2：失败重试值不值 —— 重试次数 vs 成功率 vs 期望耗时

单次成功率 p 的任务，重试 k 次的理论成功率是 `1-(1-p)^k`，但每次重试都要付出时间成本。
用蒙特卡洛验证公式，并找出「边际收益拐点」：第几次重试之后基本白干。


In [ ]:
import numpy as np
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

rng = np.random.default_rng(3)
p_list = [0.5, 0.7, 0.9]
k_max = 6
ks = np.arange(0, k_max + 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for p in p_list:
    theory = 1 - (1 - p) ** (ks + 1)
    # 蒙特卡洛：每个 trial 最多试 k+1 次，全部失败才算失败
    sim = []
    for k in ks:
        attempts = rng.random((20000, k + 1)) < p
        sim.append(attempts.any(axis=1).mean())
    axes[0].plot(ks, theory, "-", label=f"理论 p={p}")
    axes[0].plot(ks, sim, "o", ms=4)
axes[0].set_xlabel("重试次数 k"); axes[0].set_ylabel("总成功率")
axes[0].set_title("重试 vs 成功率（线=理论，点=模拟）")
axes[0].legend(); axes[0].grid(alpha=0.3)

p = 0.7
base = 10  # 单次尝试耗时(秒)
marginal, expect_t = [], []
for k in ks:
    succ = 1 - (1 - p) ** (k + 1)
    prev = 1 - (1 - p) ** k if k > 0 else 0
    marginal.append((succ - prev) * 100)
    expect_t.append(base * (1 - (1 - p) ** (k + 1)) / p)   # 期望总耗时
axes[1].bar(ks, marginal, color="#4FC3F7", label="第k次重试的边际增益(百分点)")
axes[1].plot(ks, expect_t, "r^-", label="期望总耗时(s)")
axes[1].set_xlabel("重试次数 k"); axes[1].set_title(f"边际收益递减 (p={p})")
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("观察：p=0.9 时重试 2 次成功率已达 99.9%；此后每次重试的边际增益 <0.1 个百分点。")
print("→ 重试上限通常设 2-3 次，再多是浪费配额与时间。")

## 实验 3：串行 vs 并行 —— 拓扑分层调度与关键路径

无依赖的任务本可同时跑。把 DAG 按「层」展开（每层任务并行），总耗时 = 最长依赖链（关键路径）。
用甘特图画出并行调度，并与「无脑串行」对比。


In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

tasks = {
    "店员打卡": ([], 2), "清洁": (["店员打卡"], 3), "商品检查": (["店员打卡"], 4),
    "开业": (["清洁", "商品检查"], 1), "订单处理": (["开业"], 8),
    "结算": (["开业"], 3), "数据统计": (["订单处理", "结算"], 2),
}

def earliest_finish(tasks):
    """DP 求每个任务最早完成时刻 = 关键路径调度（不限并发）"""
    memo = {}
    def fin(n):
        if n not in memo:
            deps, d = tasks[n]
            memo[n] = (max((fin(x) for x in deps), default=0)) + d
        return memo[n]
    return {n: fin(n) for n in tasks}

finish = earliest_finish(tasks)
start = {n: finish[n] - tasks[n][1] for n in tasks}
serial_time = sum(d for _, d in tasks.values())
parallel_time = max(finish.values())

fig, ax = plt.subplots(figsize=(10, 4.5))
cmap = plt.cm.Set2.colors
for i, (n, s) in enumerate(sorted(start.items(), key=lambda x: x[1])):
    d = tasks[n][1]
    ax.barh(i, d, left=s, color=cmap[i % len(cmap)], edgecolor="gray")
    ax.text(s + d / 2, i, n, ha="center", va="center", fontsize=10)
ax.set_yticks([]); ax.invert_yaxis()
ax.set_xlabel("时间 (s)")
ax.set_title(f"并行调度甘特图：总耗时 {parallel_time}s（串行需 {serial_time}s，节省 {1-parallel_time/serial_time:.0%}）")
ax.grid(axis="x", alpha=0.3); plt.tight_layout(); plt.show()

print(f"串行总耗时 = {serial_time}s，并行（关键路径）= {parallel_time}s")
print("→ 编排器给调度换来的加速上限由关键路径决定：再多的 worker 也快不过最长依赖链。")

## 实验 4：失败传播语义 —— fail-fast 还是 best-effort？

工作流里某任务失败后有两种策略：**fail-fast**（立刻终止整个流程）与 **best-effort**（跳过失败任务，无依赖的任务继续）。
给「订单处理」注入 40% 失败率，蒙特卡洛比较两种策略下「整流程完成率」与「平均完成任务数」。


In [ ]:
import random
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

DEPS = {
    "店员打卡": [], "清洁": ["店员打卡"], "商品检查": ["店员打卡"], "开业": ["清洁", "商品检查"],
    "订单处理": ["开业"], "结算": ["开业"], "数据统计": ["订单处理", "结算"],
}
FAIL_RATE = {"订单处理": 0.4}      # 只有订单处理不稳

def run(mode, rng):
    done, skipped = set(), set()
    for name in list(DEPS):        # 简化：按依赖拓扑序迭代多轮直到不动点
        pass
    order = []
    pending = set(DEPS)
    while pending:
        ready = [n for n in pending if all(d in done for d in DEPS[n])]
        if not ready: break
        for n in ready:
            pending.discard(n)
            blocked = any(d in skipped for d in DEPS[n])
            if blocked:
                skipped.add(n); continue
            if rng.random() < FAIL_RATE.get(n, 0):
                if mode == "fail_fast":
                    return len(done), False            # 整流程失败
                skipped.add(n); continue               # best_effort：跳过继续
            done.add(n)
    return len(done), len(done) == len(DEPS)

rng = random.Random(9)
N = 20000
stats = {}
for mode in ("fail_fast", "best_effort"):
    results = [run(mode, rng) for _ in range(N)]
    succ = sum(1 for _, ok in results if ok) / N
    avg_done = sum(c for c, _ in results) / N
    stats[mode] = (succ, avg_done)
    print(f"{mode:<12}: 整流程完成率 {succ:.1%}，平均完成任务 {avg_done:.2f}/7")

r = random.Random(9)
from collections import Counter
dist = Counter(run("best_effort", r)[0] for _ in range(N))
fig, ax = plt.subplots(figsize=(8, 3.8))
ks = sorted(dist)
ax.bar([str(k) for k in ks], [dist[k] / N for k in ks], color="#4DB6AC")
ax.set_xlabel("完成任务数"); ax.set_ylabel("概率")
ax.set_title("best-effort 模式完成任务数分布（订单处理失败率 40%）")
plt.tight_layout(); plt.show()

print("→ fail-fast 适合『部分结果毫无价值』的流程（如支付）；")
print("  best-effort 适合『部分结果也有用』的流程（如日报汇总：某数据源挂了照样出其余部分）。")

## 结论

| 编排能力 | 实验验证 |
|---|---|
| DAG + 拓扑排序 | 依赖显式化，顺序自动推导，环被拒绝（实验1） |
| 重试语义 | 成功率 1-(1-p)^k，边际收益 2-3 次后趋零（实验2） |
| 并行调度 | 加速上限 = 关键路径，串行 23s → 并行 17s（实验3） |
| 失败传播 | fail-fast vs best-effort 的完成率/部分可用性权衡（实验4） |

**编排引擎的本质：把『谁先谁后、失败了怎么办、能不能同时跑』从人脑搬进可测试的代码。**
→ 深入阅读：同名 `.md` 的工具编排 Level 1-4、Cron 定时任务与跨平台消息路由。
